<a href="https://colab.research.google.com/github/cyrus2281/notes/blob/main/Architecture/Architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Software Architecture Notes

>[Software Architecture Notes](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=PQ6Eg6AkHf12)

>[Event-Driven Architecture](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=TG1TXEEiVZEz)

>>[Request/Reply Pattern](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=TG1TXEEiVZEz)

>>>[Core Mechanism](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=TG1TXEEiVZEz)

>>>[Implementation Techniques](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=TG1TXEEiVZEz)

>>>>[Correlation IDs](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=TG1TXEEiVZEz)

>>>>[Temporary Queues](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=TG1TXEEiVZEz)



# Event-Driven Architecture

## Request/Reply Pattern

The **Request/Reply pattern** allows for synchronous-like behavior ("pseudo-synchronous messaging") within an asynchronous event-driven architecture.

### Core Mechanism
*   **Structure:** Utilizes two distinct queues—a **Request Queue** (for sending) and a **Reply Queue** (for receiving).
*   **Workflow:**
    1.  Sender sends a message to the Request Queue.
    2.  Sender is free to perform other processing immediately after sending (asynchronous).
    3.  Sender performs a **blocking wait** on the Reply Queue when it requires the answer.
    4.  Receiver processes the request and sends the result to the Reply Queue.
    5.  Sender retrieves the response.

---

### Implementation Techniques

#### 1. Correlation IDs
This method is used when multiple responses sit in a shared Reply Queue. It ensures the sender retrieves only the response meant for its specific request.

*   **The Problem:** The Reply Queue may contain messages intended for other senders (e.g., IDs 120, 122).
*   **The Process:**
    1.  **Sender:** Sends a request with a unique **Message ID** (e.g., 124).
    2.  **Sender:** Waits on the Reply Queue using a **Message Selector** (or filter) looking for `CorrelationID == 124`.
    3.  **Receiver:** Gets the message, processes it, and sets the response's **Correlation ID** to match the original Message ID (124).
    4.  **Receiver:** Sends the message to the Reply Queue with a new unique Message ID (e.g., 857) but the matching Correlation ID.
    5.  **Sender:** Identifies the correct message via the Correlation ID and retrieves the data.

#### 2. Temporary Queues
A simpler alternative that does not use a shared reply queue initially.

*   **The Process:**
    1.  **Sender:** Sets a **"Reply To"** header in the message indicating a temporary queue (e.g., `TemporaryQueue T1`).
    2.  **Broker:** Creates this temporary queue; it is exclusive and unknown to others.
    3.  **Sender:** Performs a blocking wait on this specific temporary queue.
    4.  **Receiver:** Sends the response directly to the temporary queue specified in the header.
    5.  **Sender:** Receives the message (no selector/filter needed since the queue is private).
    6.  **Broker:** Removes the temporary queue once the interaction is complete.

